In [1]:
import torch
import pandas as pd
import numpy as np
%load_ext autoreload
%autoreload 2

In [2]:
with open('data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print(f'lenght of the characters in the dataset {len(text)}')

lenght of the characters in the dataset 1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars = sorted(list(set(text)))
vocabulary_size = len(chars)
print('This are the single chars of the data')
print(' '.join(chars))
print(vocabulary_size)


This are the single chars of the data

   ! $ & ' , - . 3 : ; ? A B C D E F G H I J K L M N O P Q R S T U V W X Y Z a b c d e f g h i j k l m n o p q r s t u v w x y z
65


### Encoder and Decoder tokenization

In [6]:
#Encoder
stoi = {character : index for index, character in enumerate(chars)}
encode = lambda s : [stoi[character] for character in s] # encoder: take a string, output a list of integers


#Decoder
itos = {index : character for index, character in enumerate(chars)}
decode = lambda l : ''.join([itos[index] for index in l])

In [7]:
print(encode('Hi there my name is Manuel'))
print(decode(encode('Hi there my name is Manuel')))

[20, 47, 1, 58, 46, 43, 56, 43, 1, 51, 63, 1, 52, 39, 51, 43, 1, 47, 57, 1, 25, 39, 52, 59, 43, 50]
Hi there my name is Manuel


### Now encode all the dataset in the for using torch

In [8]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:10])
print(decode(encode(text[:10])))

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])
First Citi


### Divide in training and validation

In [9]:
n_len = int(0.90 * len(data))
train_data = data[:n_len]
val_data = data[n_len:]
len(train_data), len(val_data)

(1003854, 111540)

In [10]:
sequence_lenght = 8
train_data[:sequence_lenght + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
input_x = train_data[ :sequence_lenght]
target_y = train_data[1: sequence_lenght + 1 ]

for c in range(sequence_lenght):
    input = input_x[: c + 1]
    target = target_y[c]
    print(f'This is the input {input} this is the target {target}')

This is the input tensor([18]) this is the target 47
This is the input tensor([18, 47]) this is the target 56
This is the input tensor([18, 47, 56]) this is the target 57
This is the input tensor([18, 47, 56, 57]) this is the target 58
This is the input tensor([18, 47, 56, 57, 58]) this is the target 1
This is the input tensor([18, 47, 56, 57, 58,  1]) this is the target 15
This is the input tensor([18, 47, 56, 57, 58,  1, 15]) this is the target 47
This is the input tensor([18, 47, 56, 57, 58,  1, 15, 47]) this is the target 58


In [12]:
from utils import get_batch


In [30]:
batch_size = 4 # how many independent sequences will we process in parallel?
sequence_lenght = 8 # What is the maximum context lenght for the prediction? 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

xb, yb = get_batch(
    split='train',
    batch_size=batch_size,
    sequence_length=sequence_lenght,
    train_data=train_data,
    val_data=val_data,
    device=device)


print("The inputs")
print(xb.shape)
print(xb)

print("The targets")
print(yb.shape)
print(yb)


The inputs
torch.Size([4, 8])
tensor([[53, 52, 43,  1, 57, 53,  1, 40],
        [46, 53, 59,  6,  1, 40, 43, 47],
        [35, 46, 39, 58,  1, 52, 43, 61],
        [ 1, 39, 63,  6,  1, 46, 47, 57]])
The targets
torch.Size([4, 8])
tensor([[52, 43,  1, 57, 53,  1, 40, 39],
        [53, 59,  6,  1, 40, 43, 47, 52],
        [46, 39, 58,  1, 52, 43, 61, 57],
        [39, 63,  6,  1, 46, 47, 57,  1]])


In [32]:
for b in range(batch_size):
    for sl in range(sequence_lenght):
        context = xb[b, :sl+1]
        target = yb[b, sl]
        print(f'This is the input {context.tolist()} this is the target {target}')
    break

This is the input [53] this is the target 52
This is the input [53, 52] this is the target 43
This is the input [53, 52, 43] this is the target 1
This is the input [53, 52, 43, 1] this is the target 57
This is the input [53, 52, 43, 1, 57] this is the target 53
This is the input [53, 52, 43, 1, 57, 53] this is the target 1
This is the input [53, 52, 43, 1, 57, 53, 1] this is the target 40
This is the input [53, 52, 43, 1, 57, 53, 1, 40] this is the target 39


In [34]:
print(f'The single input for the transformer \n {xb}')

The single input for the transformer 
 tensor([[53, 52, 43,  1, 57, 53,  1, 40],
        [46, 53, 59,  6,  1, 40, 43, 47],
        [35, 46, 39, 58,  1, 52, 43, 61],
        [ 1, 39, 63,  6,  1, 46, 47, 57]])


In [36]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocabulary_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.7015, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


In [37]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

/Users/administrador/Library/Python/3.11/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [41]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb =get_batch(
    split='train',
    batch_size=batch_size,
    sequence_length=sequence_lenght,
    train_data=train_data,
    val_data=val_data,
    device=device)

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


2.4618213176727295


In [42]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


CE:
FOFoure pprour nearouay Benofo leares featantithe in, tlint;
he ncorung h f

To a, h hies ave pewhofeyoureat.'s al he
Whet wse llofoupat thotin
BO:
D gh VII dd y LAULI m ithe;

NGBAlar to must pate'BUnd tepuleavex on tane m, mpe agurif, wes BED an RENTHareas r supe whewrd rever a,
Toverimarstouaseasthe st ar whedicth--niverdan n?
BUKE:
NCKELAn themethe:
We-cecoft s hoavencemm wialllans, o y.
Ficheak ee dl at; howoughe linmy age cither t hatsancorepld o vo:
Tom surill WA:
HONam the ngu fe s w


### Scaled Dot-product Attention

In [72]:
torch.manual_seed(1337)

B, T, C = 4, 8, 32 # batch, time, channels
x = torch.randn(B, T, C)


head_size = 16
queries = nn.Linear(C, head_size, bias=False) # What im looking for?
keys = nn.Linear(C, head_size, bias=False) #What do I contain?
values = nn.Linear(C, head_size, bias=False) #If you make attention to me, this is the real information I will give you

q = queries(x) # B, t, 16
k = keys(x) # B, t, 16
v = values(x) # B, t, 16
wei = (q @ k.transpose(-2, -1)) / (head_size ** 0.5) # B, T, 16 @ B, 16, T -> B, T, T


tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
print(wei[0])

output = wei @ v
output[0] #If the score is high == means a lot of attention

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5221, 0.4779, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3602, 0.3210, 0.3188, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2980, 0.4039, 0.1578, 0.1404, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1643, 0.1243, 0.1678, 0.1865, 0.3570, 0.0000, 0.0000, 0.0000],
        [0.2656, 0.2110, 0.1137, 0.1214, 0.2018, 0.0865, 0.0000, 0.0000],
        [0.1761, 0.1327, 0.1371, 0.0974, 0.1476, 0.1918, 0.1173, 0.0000],
        [0.1046, 0.1260, 0.0922, 0.0906, 0.1476, 0.1588, 0.1432, 0.1371]],
       grad_fn=<SelectBackward0>)


tensor([[-0.1571,  0.8801,  0.1615, -0.7824, -0.1429,  0.7468,  0.1007, -0.5239,
         -0.8873,  0.1907,  0.1762, -0.5943, -0.4812, -0.4860,  0.2862,  0.5710],
        [ 0.3156,  0.0704, -0.0706, -0.1604, -0.1344,  0.1558, -0.2001, -0.2886,
         -0.4120,  0.4947,  0.4806, -0.3233, -0.0231, -0.0158,  0.0837,  0.9683],
        [ 0.4029, -0.0241, -0.2422,  0.0145,  0.0144, -0.0129, -0.0916, -0.1296,
         -0.3266,  0.0527,  0.3795, -0.0745, -0.1562, -0.0397, -0.0320,  1.0982],
        [ 0.4779, -0.2057, -0.2656,  0.1018,  0.0853, -0.1675, -0.1532, -0.1084,
         -0.1658,  0.2856,  0.3110, -0.0448,  0.0502,  0.1370,  0.1077,  0.9660],
        [ 0.3579,  0.2417,  0.0711,  0.1706,  0.2915,  0.1982,  0.2582,  0.1424,
         -0.3187, -0.4826, -0.1465, -0.0625, -0.4228,  0.1282,  0.0811,  0.6980],
        [ 0.2370,  0.1631, -0.0279,  0.1223,  0.1766,  0.1643,  0.0528, -0.0590,
         -0.2692, -0.0797,  0.0590, -0.0432, -0.2163,  0.0780,  0.1760,  0.6983],
        [ 0.0250,  0.1